In [33]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

In [34]:
train_df = pd.read_csv('train.csv')[['text', 'target']]
test_df = pd.read_csv('test.csv')[['text', 'id']]

#tokenizer init
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }
    
train_dataset = TweetDataset(train_df['text'].tolist(), train_df['target'].tolist(), tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased')
devices = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model.to(devices)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9640.53it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [35]:
#optimization and training
optimizer = AdamW(model.parameters(), lr=2e-5)

print("starting training...")
epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(devices)
        attention_mask = batch['attention_mask'].to(devices)
        labels = batch['labels'].to(devices)

        optimizer.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = output.loss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}')



starting training...
Epoch 1/3, Loss: 0.001449311981682016
Epoch 1/3, Loss: 0.002893194556236267
Epoch 1/3, Loss: 0.004323516448004907
Epoch 1/3, Loss: 0.0056393100183551054
Epoch 1/3, Loss: 0.007022238203457424
Epoch 1/3, Loss: 0.008338659626095235
Epoch 1/3, Loss: 0.009839108385959593
Epoch 1/3, Loss: 0.011208985157373572
Epoch 1/3, Loss: 0.012524691073834394
Epoch 1/3, Loss: 0.013971718169060074
Epoch 1/3, Loss: 0.015144067652085248
Epoch 1/3, Loss: 0.016351235513927555
Epoch 1/3, Loss: 0.017630275307583206
Epoch 1/3, Loss: 0.019016022942647214
Epoch 1/3, Loss: 0.020292797384141872
Epoch 1/3, Loss: 0.021566125650365812
Epoch 1/3, Loss: 0.022815264573618144
Epoch 1/3, Loss: 0.023994371670634805
Epoch 1/3, Loss: 0.02509657700522607
Epoch 1/3, Loss: 0.02622578261780138
Epoch 1/3, Loss: 0.02748411543229047
Epoch 1/3, Loss: 0.028391993972433714
Epoch 1/3, Loss: 0.02957782575062343
Epoch 1/3, Loss: 0.030522323769180716
Epoch 1/3, Loss: 0.03148952946692955
Epoch 1/3, Loss: 0.03284395761600

In [39]:
#dataloader for test
import numpy as np

class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

test_dataset = TestDataset(test_df['text'].tolist(), tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model.eval()
prediction = []

print("Starting NLP Predictions...")

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(devices)
        attention_mask = batch['attention_mask'].to(devices)

        output = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = output.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        prediction.extend(preds)
print("Predictions completed.")




Starting NLP Predictions...
Predictions completed.


In [40]:
submission = pd.DataFrame({
   'id': test_df['id'],
   'target': prediction
})

submission.to_csv('nlp_disaster_submission.csv', index=False)
print("Saved to nlp_disaster_submission.csv")

Saved to nlp_disaster_submission.csv
